In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DSMarket — Tarea 4: Propuesta de sistema de reposición inteligente de stock

<div style="background-color:#2D8C4E;border-left:6px solid #2D8C4E;padding:14px;border-radius:10px">

**Objetivo del notebook**

Traducir el modelo de forecasting desarrollado en la Tarea 3 a una propuesta operativa de reposición de stock para DSMarket.

Este notebook no implementa un sistema productivo completo, sino que presenta una solución razonada y defendible para:

- transformar predicciones diarias en decisiones semanales de pedido,
- reducir roturas de stock mediante calibración y safety stock,
- y definir una arquitectura mínima de despliegue, API y monitorización.

</div>

<a id="indice"></a>

## Índice

1. [Contexto y comunicación interna](#contexto)
2. [Caso de uso: reposición semanal](#caso-uso)
3. [Diseño operativo de la reposición](#diseno-reposicion)
4. [Extensiones necesarias al modelo](#extensiones)
5. [Productivización y API](#api)
6. [Conclusión ejecutiva](#cierre)
7. 

## Cómo leer este notebook

La lógica del notebook sigue esta secuencia:

1. recordar el contexto de negocio y el resultado del forecasting,
2. definir el problema operativo de reposición,
3. proponer cómo convertir la predicción en una orden semanal,
4. identificar qué mejoras técnicas son necesarias antes de producción,
5. y cerrar con una propuesta ejecutiva para negocio y tecnología.

La idea no es construir aquí un sistema completo en producción, sino presentar una propuesta operativa realista, conectada con los resultados del proyecto.

[⬆ Volver al índice](#indice)

<a id="contexto"></a>

## Contexto y comunicación interna

En esta tarea paso del plano predictivo al plano operativo.

Después de las fases anteriores del proyecto, la conclusión relevante es la siguiente:

- **E2** fue el mejor modelo desde el punto de vista técnico.
- **E2 calibrado** fue la mejor solución desde el punto de vista de negocio, porque corrige la infra-predicción y reduce el riesgo de rotura de stock.

A partir de esa base, este notebook propone cómo usar ese resultado para apoyar una reposición semanal más inteligente.

[⬆ Volver al índice](#indice)

## Sección 1 — Comunicación interna

**De:** Nicole Chen, Data Scientist Senior  
**Para:** Paul Rogers, CFO  
**CC:** Michelle Huggins, CDO  
**Asunto:** Propuesta de sistema de reposición semanal basado en forecasting — DSMarket  
**Fecha:** Marzo 2025

---

Paul,

Tal como acordamos, presento la propuesta para aplicar los modelos de predicción desarrollados en las fases anteriores del proyecto a la reposición de stock en las tiendas de Nueva York, Boston y Philadelphia.

Durante las últimas semanas se completaron tres bloques de trabajo:

- **EDA y series temporales** — Se identificaron patrones de demanda por tienda, ciudad y categoría, así como segmentos con mayor riesgo e impacto operativo.
- **Clustering** — Se segmentaron los productos en grupos con comportamientos de demanda distintos, útiles como capa contextual de análisis.
- **Forecasting** — Se construyó un sistema de predicción donde **E2** fue la mejor opción técnica y **E2 calibrado** la mejor opción operativa, al corregir el sesgo infra-predictivo detectado en el período de test.

A partir de estos resultados, esta propuesta detalla:

1. cómo traducir las predicciones diarias en órdenes de reposición semanales,
2. qué elementos adicionales son necesarios para reducir roturas de stock,
3. qué extensiones técnicas conviene abordar antes de un despliegue robusto,
4. y cómo exponer la solución mediante API y monitorización.

Quedo disponible para revisar esta propuesta antes de la presentación conjunta con negocio y tecnología.

Nicole

[⬆ Volver al índice](#indice)

<a id="caso-uso"></a>

## Caso de uso: reposición semanal

El objetivo de esta propuesta es transformar el forecasting en una decisión operativa útil.

La pregunta ya no es cuál es el mejor modelo en términos puramente predictivos, sino:

**cómo usar la predicción para reducir roturas de stock sin generar un exceso de inventario innecesario.**

[⬆ Volver al índice](#indice)

### 2.1 El problema operativo

DSMarket gestiona actualmente la reposición de stock de forma manual o mediante reglas fijas, por ejemplo reponer cuando el stock cae por debajo de un umbral predefinido.

Este enfoque genera dos problemas recurrentes:

- **Roturas de stock** en productos de alta demanda o demanda volátil, con pérdida directa de ventas y deterioro de experiencia de cliente.
- **Exceso de stock** en productos lentos o estacionales, con coste de almacenamiento, inmovilización de capital y mayor riesgo de obsolescencia o caducidad.

La oportunidad del proyecto consiste en sustituir ese enfoque por uno basado en predicción de demanda y control explícito del riesgo operativo.

La idea no es pedir “lo mismo que se prevé vender”, sino construir una recomendación de pedido que combine:

- demanda esperada,
- corrección del sesgo del modelo,
- stock de seguridad,
- y stock disponible.

[⬆ Volver al índice](#indice)


### 2.2 Objetivo de la propuesta

La propuesta de reposición inteligente persigue cuatro objetivos de negocio:

1. **Reducir roturas de stock** en productos y tiendas con mayor riesgo operativo.
2. **Disminuir exceso de inventario** en artículos de baja rotación o comportamiento estacional.
3. **Sustituir reglas fijas por decisiones cuantitativas**, apoyadas en forecasting y error real observado.
4. **Preparar una base productivizable**, consumible por negocio y por sistemas internos.

En términos prácticos, esta propuesta busca responder a una pregunta muy simple:

**¿Cuánto debería pedir cada tienda para cada producto la próxima semana, dadas las ventas esperadas y el riesgo de quedarse corta?**

[⬆ Volver al índice](#indice)

<a id="diseno-reposicion"></a>

## Diseño operativo de la reposición

En esta sección traduzco el resultado del forecasting a una lógica concreta de pedido.

La idea central es que una predicción útil para negocio no debe convertirse directamente en una orden de compra,
sino pasar por una capa adicional de corrección y protección frente al riesgo operativo.

[⬆ Volver al índice](#indice)

### 3.1 Lógica operativa propuesta

La propuesta no consiste en convertir directamente la predicción en una orden de compra.

Para reducir el riesgo de rotura de stock, la reposición debe construirse como una combinación de tres elementos:

1. **Demanda esperada de los próximos 7 días**, obtenida a partir del forecasting diario agregado.
2. **Safety stock**, calculado en función del error histórico del modelo y del nivel de servicio deseado.
3. **Stock disponible en tienda**, descontado en el momento de generar la orden.

De esta forma, el sistema deja de depender de reglas fijas o umbrales arbitrarios
y pasa a basarse en una previsión cuantitativa ajustada al comportamiento real de la demanda.

La decisión importante aquí es que la reposición ya no responde solo a cuánto se espera vender,
sino también a cuánto riesgo se está dispuesto a asumir.

[⬆ Volver al índice](#indice)

### 3.2 De predicción diaria a orden semanal

El modelo genera predicciones a nivel **tienda × producto × día**.  
Para la reposición semanal, estas predicciones se traducen a demanda operativa en dos pasos.

**Paso 1 — Agregación semanal de la predicción diaria**

Para cada combinación tienda × producto, se suman los 7 días del horizonte:

`Demanda_estimada_7d = Σ predicción_día_i (i = 1..7)`

**Paso 2 — Aplicación de la calibración**

Dado que el modelo E2 mostró una infra-predicción sistemática en el período de test,  
se aplica un factor global de calibración obtenido a partir del ratio predicho/real:

`Demanda_calibrada_7d = Demanda_estimada_7d × 1.3657`

Esta cantidad representa la mejor estimación operativa de la demanda semanal esperada.

La decisión importante aquí es que la reposición no se apoya en la predicción bruta del modelo,
sino en su versión calibrada, que reduce el riesgo de infraabastecimiento.

[⬆ Volver al índice](#indice)

### 3.3 Safety stock y nivel de servicio

Incluso con la calibración, la predicción sigue estando sujeta a incertidumbre.

Por eso, la propuesta incorpora una capa de **safety stock** que actúa como colchón frente a:

- error residual del modelo,
- variabilidad real de la demanda,
- y riesgo de quedarse corto en artículos críticos.

La lógica del safety stock debe depender de dos elementos:

1. **Variabilidad del producto o segmento**
2. **Nivel de servicio objetivo**

Esto permite ajustar la agresividad de la reposición según contexto de negocio.

Ejemplos de nivel de servicio objetivo:

- **90%** → menor stock de seguridad, mayor eficiencia de inventario
- **95%** → equilibrio razonable entre coste y riesgo
- **99%** → enfoque conservador para productos críticos

La implicación importante es que el forecast no sustituye al stock de seguridad.
Lo complementa.

[⬆ Volver al índice](#indice)

### 3.4 Fórmula propuesta de safety stock

Una forma simple y defendible de construir el stock de seguridad es basarse en el error histórico del modelo:

`Safety_stock = z × σ_error × sqrt(L)`

Donde:

- **z** representa el nivel de servicio deseado,
- **σ_error** representa la desviación del error histórico,
- **L** representa el lead time o período de cobertura considerado.

En una versión inicial del sistema, este cálculo puede hacerse a nivel de:

- categoría,
- clúster de producto,
- o tienda × categoría,

según el nivel de granularidad que resulte más estable.

La ventaja de esta aproximación es que el safety stock deja de ser una regla arbitraria
y pasa a estar anclado en el comportamiento real del error del modelo.

[⬆ Volver al índice](#indice)

### 3.5 Fórmula final del pedido recomendado

La lógica operativa propuesta para cada combinación tienda × producto es la siguiente:

`Pedido_recomendado = max(0, Demanda_calibrada_7d + Safety_stock - Stock_disponible)`

Donde:

- **Demanda_calibrada_7d** recoge la mejor estimación operativa de ventas para la semana siguiente.
- **Safety_stock** protege frente a incertidumbre del modelo y variabilidad de la demanda.
- **Stock_disponible** representa el inventario utilizable en el momento de lanzar la orden.

Esta formulación separa correctamente dos problemas distintos:

- **forecasting**, que estima la demanda esperada,
- y **reposición**, que incorpora además la tolerancia al riesgo y el nivel de servicio objetivo.

Desde el punto de vista de negocio, esta es la pieza clave de la propuesta:
el pedido ya no se construye sobre intuición o reglas fijas,
sino sobre una combinación explícita de previsión, error y stock disponible.

[⬆ Volver al índice](#indice)

📌 **Conclusión / Decisión**

La propuesta de reposición no utiliza la predicción diaria de forma directa,
sino que la transforma en una estimación semanal calibrada y protegida con safety stock.

Esto permite convertir el resultado del forecasting en una decisión operativa más robusta,
especialmente en un contexto donde la infra-predicción del modelo generaría roturas de stock
si se utilizara sin corrección adicional.

[⬆ Volver al índice](#indice)

<a id="extensiones"></a>

## Extensiones necesarias al modelo

El sistema propuesto es utilizable en un piloto inicial con **E2 calibrado**.

Sin embargo, antes de un despliegue robusto a escala conviene incorporar varias mejoras
que aumentarían la precisión, reducirían el sesgo y harían la solución más estable desde el punto de vista operativo.

[⬆ Volver al índice](#indice)


### 4.1 Deuda técnica identificada

El modelo E2 calibrado es funcional y está validado, pero fue diseñado como una solución de forecasting aplicada en entorno notebook.

Antes de usarlo como sistema de reposición de stock en producción, identificamos cuatro mejoras concretas que aumentarían su robustez:

1. **manejo más adecuado de zero inflation**,  
2. **mayor sensibilidad a la identidad de la serie**,  
3. **calibración más fina que el factor global actual**,  
4. **capacidad de estimar incertidumbre y no solo predicción puntual**.

Estas mejoras no son imprescindibles para un piloto acotado,
pero sí recomendables para un despliegue operativo más sólido.

[⬆ Volver al índice](#indice)

### 4.2 E5 — Objetivo Tweedie para zero inflation

El **56.3%** de los registros del dataset tienen ventas iguales a cero.

El modelo actual (`objective = regression_l1`) optimiza la mediana y tiende a comportarse de forma conservadora,
lo que ayuda a explicar parte del sesgo infra-predictivo observado antes de calibrar.

Una mejora natural sería entrenar una variante con **objetivo Tweedie**,
una distribución diseñada para datos con exceso de ceros y valores positivos.

### Impacto esperado

- mejor ajuste en series de baja rotación,
- menor necesidad de corrección global en postproceso,
- y mayor coherencia entre la distribución del target y la función objetivo del modelo.

### Lectura práctica

Esta mejora no sustituye a la calibración por sí sola,
pero sí podría reducir el sesgo estructural del modelo desde el entrenamiento.

[⬆ Volver al índice](#indice)

### 4.3 E6 — Identidad de serie como feature

El modelo actual no incorpora de forma explícita la identidad de la serie que está prediciendo.

Añadir variables categóricas como:

- `store_code`
- `category`
- `city`

permitiría al modelo aprender diferencias estructurales entre contextos de demanda.

Esto es especialmente relevante porque en la Tarea 3 se documentó un sesgo distinto por categoría:

- **ACCESSORIES** → sesgo más negativo
- **HOME & GARDEN** → sesgo intermedio
- **SUPERMARKET** → sesgo menos severo, pero todavía relevante

### Impacto esperado

- menor bias por segmento,
- mejor ajuste a heterogeneidad entre tiendas y categorías,
- y menos dependencia de una calibración global uniforme.

La hipótesis aquí es sencilla:
parte del error actual no viene solo del nivel de ventas,
sino de que el modelo no distingue suficientemente bien el contexto que está prediciendo.

[⬆ Volver al índice](#indice)

### 4.4 E7 — Calibración por categoría

El factor de calibración actual (**×1.3657**) se aplica de forma global a todos los productos y tiendas.

Esto es útil como primera corrección operativa,
pero simplifica en exceso una realidad donde el sesgo varía de forma importante por categoría.

La mejora más inmediata y de menor coste consiste en calcular un factor de calibración independiente para cada categoría.

### Factores correctores estimados

| Categoría | Bias previo aproximado | Factor corrector estimado |
|---|---:|---:|
| ACCESSORIES | más negativo | ×1.82 |
| HOME & GARDEN | intermedio | ×1.51 |
| SUPERMARKET | menos severo | ×1.27 |

### Impacto esperado

- corrección más fina del sesgo,
- menor infra-predicción en los segmentos más problemáticos,
- y menor dependencia de un único multiplicador global.

### Lectura práctica

Esta es la extensión con mejor relación entre impacto y esfuerzo.

Por eso, si hubiera que priorizar una sola mejora antes del primer despliegue piloto,
la recomendación sería **implementar E7 primero**.

[⬆ Volver al índice](#indice)

### 4.5 E8 — Intervalos de predicción y cuantiles

Para reposición con riesgo controlado, una predicción puntual no siempre es suficiente.

Desde negocio interesa poder elegir entre distintos escenarios de cobertura, por ejemplo:

- una reposición más agresiva en coste,
- una reposición equilibrada,
- o una reposición más conservadora para evitar roturas.

Una forma de hacerlo es entrenar modelos de **regresión cuantílica** que devuelvan varios niveles de predicción:

- **Q10** → escenario optimista
- **Q50** → predicción central
- **Q90** → escenario conservador

### Impacto esperado

- capacidad de traducir el forecasting a niveles de servicio explícitos,
- mejor alineación entre analítica y decisiones de inventario,
- y una política de reposición más flexible según categoría o criticidad.

### Lectura práctica

Esta extensión es especialmente valiosa,
pero también más exigente en diseño y validación que las anteriores.

Por ello, se plantea como mejora de medio plazo.

[⬆ Volver al índice](#indice)

### 4.6 Priorización de extensiones

| Extensión | Impacto esperado | Esfuerzo estimado | Prioridad |
|---|---|---|---|
| E7 — Calibración por categoría | Alto | Bajo | **Inmediata** |
| E5 — Objetivo Tweedie | Alto | Medio | Corto plazo |
| E6 — Identidad de serie | Medio | Medio | Corto plazo |
| E8 — Cuantiles / intervalos | Alto | Alto | Medio plazo |

### Recomendación

La ruta más realista para DSMarket sería:

1. **Piloto inicial con E2 calibrado**
2. **Implementación inmediata de E7**
3. **Experimentación posterior con E5 y E6**
4. **Diseño de una solución basada en cuantiles en una fase posterior**

De esta forma, la empresa puede capturar valor operativo pronto
sin renunciar a una mejora técnica progresiva del sistema.

[⬆ Volver al índice](#indice)

📌 **Conclusión / Decisión**

La solución propuesta puede utilizarse como base de un piloto,
pero un despliegue robusto a escala requiere reducir el sesgo estructural del modelo
y mejorar la sensibilidad al contexto de cada serie.

La mejora más inmediata y rentable sería la **calibración por categoría**,
mientras que Tweedie, identidad de serie y cuantiles constituyen el roadmap natural de evolución técnica.

[⬆ Volver al índice](#indice)

<a id="api"></a>

## Productivización y API

Una propuesta de reposición no es útil si depende de ejecución manual en notebook.

Por eso, esta sección traduce la lógica analítica a una arquitectura mínima de uso operativo,
pensada para que negocio y sistemas internos puedan consumir la recomendación de pedido
sin depender del detalle técnico del modelo.

[⬆ Volver al índice](#indice)


### 5.1 Visión general

Pasar del modelo entrenado en entorno notebook a una solución operativa requiere tres componentes:

1. **Pipeline de actualización y reentrenamiento**, para incorporar datos recientes.
2. **API de predicción**, para exponer la recomendación de pedido como servicio.
3. **Sistema de monitorización**, para detectar degradación del modelo y activar recalibración o revisión.

Estos tres componentes forman el esqueleto mínimo de una solución MLOps para DSMarket.

La idea no es desplegar una plataforma compleja desde el primer día,
sino definir una arquitectura progresiva y realista que permita pasar del piloto a operación.

[⬆ Volver al índice](#indice)

### 5.2 Pipeline de reentrenamiento periódico

El modelo debe actualizarse para incorporar los patrones más recientes de demanda
y evitar que la solución pierda calidad con el tiempo.

### Propuesta de cadencia

| Proceso | Frecuencia | Detalle |
|---|---|---|
| Reentrenamiento completo | Mensual | Nuevo modelo con ventana deslizante de 2 años |
| Recalibración | Mensual | Actualizar factor global o por categoría |
| Actualización de features | Semanal | Incorporar últimas ventas reales y variables exógenas |
| Validación automática | En cada reentrenamiento | MAE y bias sobre las últimas 4 semanas |

### Criterio de rechazo del nuevo modelo

Si el modelo reentrenado empeora de forma material respecto al modelo en producción,
el pipeline no debe promoverlo automáticamente.

Una regla simple y defendible sería:

- rechazar el nuevo modelo si el **MAE** empeora más de un **10%**
- o si el **bias** se aleja significativamente del rango aceptable observado en producción

En ese caso, el sistema mantiene la versión anterior y genera una alerta para revisión.

[⬆ Volver al índice](#indice)

### 5.3 Stack recomendado para el pipeline

Para una primera versión operativa, la arquitectura recomendada sería la siguiente:

- **Orquestación:** Apache Airflow o Prefect
- **Registro de modelos:** MLflow
- **Almacenamiento de datos:** warehouse corporativo de DSMarket
- **Contenerización:** Docker
- **Ejecución programada:** entorno cloud o servidor interno con scheduler

La elección concreta entre estas herramientas no cambia la lógica del sistema.
Lo importante es que la solución permita:

- versionar modelos,
- repetir el entrenamiento de forma controlada,
- y conservar trazabilidad de métricas y decisiones de promoción.

[⬆ Volver al índice](#indice)

### 5.4 Diseño de la API (requerimiento de Martin)

La API expone las predicciones como servicio REST,
de forma que cualquier sistema interno pueda consultar recomendaciones de pedido
sin depender del detalle técnico del modelo.

### Endpoint principal

`POST /api/v1/forecast`

### Request de ejemplo

```json
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "horizon_days": 7,
  "service_level": 0.95
}

--- JSON ---
{
  "store_id": "NYC_3",
  "item_id": "PROD_00142",
  "forecast_units_7d": 140,
  "safety_stock": 35,
  "recommended_order": 175,
  "model_version": "E2_calibrated_v1",
  "confidence": "high"
}


### 5.5 Endpoints secundarios

| Endpoint | Método | Descripción |
|---|---|---|
| `/api/v1/forecast/batch` | POST | Predicción para múltiples productos |
| `/api/v1/forecast/store/{store_id}` | GET | Pedidos recomendados para una tienda completa |
| `/api/v1/model/metrics` | GET | Métricas actuales del modelo en producción |
| `/api/v1/model/status` | GET | Versión activa, fecha de último reentrenamiento y estado |

### Stack recomendado

- **Framework API:** FastAPI
- **Contenerización:** Docker
- **Despliegue:** AWS o Azure
- **Autenticación:** token interno o gateway corporativo

La recomendación aquí no es construir una arquitectura compleja desde el inicio,
sino una API pequeña, clara y mantenible.

[⬆ Volver al índice](#indice)

### 5.6 Monitorización y alertas

Una vez desplegado, el modelo debe vigilarse de forma continua.

La métrica más importante para este caso de uso no es solo el error medio,
sino la deriva entre demanda real y predicción, ya que una infra-predicción sostenida
se traduce directamente en riesgo de rotura de stock.

### Métricas clave a monitorizar

| Métrica | Frecuencia | Umbral orientativo de alerta |
|---|---|---|
| Ratio predicho/real global | Semanal | Fuera del rango 0.90 – 1.10 |
| Ratio por categoría | Semanal | Fuera del rango 0.85 – 1.15 |
| MAE rolling 4 semanas | Semanal | Incremento > 15% sobre baseline |
| % roturas de stock reales | Semanal | > 5% de combinaciones tienda × producto |

### Regla operativa recomendada

Si el ratio global o por categoría se desvía de forma persistente durante varias semanas,
el sistema debe lanzar una alerta y proponer una recalibración o revisión del modelo.

Esto evita tratar como ruido un problema que en realidad puede convertirse rápidamente
en pérdida de ventas o en sobrestock.

[⬆ Volver al índice](#indice)

### 5.7 Recalibración y mantenimiento

La calibración no debe entenderse como un ajuste puntual e inmutable.

En un entorno real, el factor corrector debe revisarse periódicamente,
porque puede cambiar si cambia:

- el mix de producto,
- la intensidad promocional,
- el comportamiento de las categorías,
- o la calidad base del modelo.

### Recomendación práctica

La primera versión del sistema puede funcionar con:

- **E2 calibrado**
- recalibración mensual
- y monitorización semanal del ratio predicho/real

En una fase posterior, esta lógica debería evolucionar hacia:

- calibración por categoría,
- validación automática de estabilidad,
- y eventualmente modelos con cuantiles para soportar decisiones según nivel de servicio.

[⬆ Volver al índice](#indice)

📌 **Conclusión / Decisión**

La solución propuesta no necesita una plataforma compleja para arrancar,
pero sí una estructura mínima que permita actualizar el modelo, exponer la recomendación de pedido y vigilar su calidad en producción.

La combinación de pipeline periódico, API ligera y monitorización continua
es suficiente para convertir el trabajo analítico en una solución operativa defendible para DSMarket.

[⬆ Volver al índice](#indice)

<a id="cierre"></a>

## Conclusión ejecutiva

En esta última sección sintetizo la propuesta desde el punto de vista de negocio y traduzco el trabajo técnico previo a una recomendación operativa clara para DSMarket.

[⬆ Volver al índice](#indice)

### 6.1 Resumen para Paul Rogers, CFO

Este documento presenta una propuesta para convertir el trabajo de forecasting desarrollado en las Tareas 1 a 3 en una solución operativa de reposición de stock para DSMarket.

### Lo que existe hoy

- Una solución de forecasting validada sobre horizonte de 28 días.
- Un modelo donde **E2** fue la mejor opción técnica.
- Una versión **E2 calibrada** que corrige el sesgo infra-predictivo y resulta más adecuada para uso operativo.
- Cobertura completa sobre las combinaciones **tienda × producto** del negocio analizado.

### Lo que esta propuesta habilita

- Cálculo automático del pedido semanal para cada combinación tienda × producto.
- Uso de **safety stock** basado en error real del modelo, no en reglas arbitrarias.
- Posibilidad de definir niveles de servicio distintos según riesgo, categoría o contexto operativo.
- Exposición del sistema mediante API, consumible por otras áreas o sistemas internos.

### Qué queda antes de un despliegue robusto

| Acción | Responsable sugerido | Plazo orientativo |
|---|---|---|
| E7 — Calibración por categoría | Data Science | 1 semana |
| Diseño y despliegue inicial de API | Data Science + Tecnología | 2–3 semanas |
| Configuración de monitorización | Data Science + Operaciones | 1–2 semanas |
| Piloto controlado en tiendas seleccionadas | Operaciones + Negocio | 4 semanas |

[⬆ Volver al índice](#indice)

### 6.2 Próximos pasos recomendados

La ruta recomendada para DSMarket sería la siguiente:

1. **Aprobar esta propuesta** en reunión conjunta entre negocio, datos y tecnología.
2. **Implementar E7** (calibración por categoría) como mejora inmediata de bajo coste.
3. **Desarrollar la API** para exponer la recomendación de pedido semanal.
4. **Lanzar un piloto controlado en 2 tiendas**, una de alto volumen y otra de comportamiento más volátil.
5. **Revisar los resultados del piloto** tras 4 semanas y decidir el paso a despliegue ampliado.

Esta secuencia permite capturar valor operativo pronto sin exigir desde el inicio una infraestructura excesivamente compleja.

[⬆ Volver al índice](#indice)

## Memorando ejecutivo — Respuesta a la Tarea 4

<div style="background-color:#264653;border-left:6px solid #264653;padding:14px;border-radius:10px">

**Asunto:** Propuesta de reposición inteligente basada en forecasting para DSMarket

La conclusión de esta tarea es que DSMarket ya dispone de una base analítica suficiente para pilotar un sistema de reposición más inteligente que el actual.

La solución recomendada no consiste en usar directamente la predicción del modelo,
sino en combinar:

- **forecast calibrado**,  
- **safety stock**,  
- **stock disponible**,  
- y **monitorización continua**.

Desde el punto de vista técnico, **E2** fue el mejor modelo.
Desde el punto de vista operativo, la mejor alternativa es **E2 calibrado**,
porque reduce el sesgo infra-predictivo y hace más segura la decisión de reposición.

La propuesta es viable como piloto y además deja identificado un roadmap claro de evolución:

- calibración más fina por categoría,
- mejora del modelo frente a zero inflation,
- y despliegue mediante API y pipeline de mantenimiento.

En consecuencia, la recomendación es avanzar con un **piloto controlado**,
medir su impacto real en roturas y sobrestock,
y utilizar ese aprendizaje como base para un despliegue progresivo.

</div>

[⬆ Volver al índice](#indice)

---

*Documento elaborado por Nicole Chen, Data Scientist Senior — DSMarket*  
*Marzo 2025*